# 01 - Train RF models and compute SHAP

This notebook is the authoritative model-training workflow. It consumes the two feature
matrices from notebook 00, the supplied FUCCI HDF5 store, and the bundled cell-curation
lists. It generates every RF, residual, prediction, permutation-sanity-check, and SHAP
table required by the two downstream publication notebooks.

Run from the `04_Feature_analysis` directory after notebook 00. The complete run is intentionally
expensive: it performs leave-one-batch-out fitting for 2,000 genes and SHAP extraction.
All regenerated artifacts are staged in `outputs/rerun/`; this notebook never replaces the
current publication artifacts in `outputs/`.


## Reproducibility and runtime notes

This workflow is expensive. The all-feature analysis fits 1,000-tree random forests for 2,000 genes across six leave-one-batch-out folds; the 150 individual-gene comparisons add further 1,000-tree fits. Run the notebook only when regenerating publication training artifacts.

The four subset RFs are feature-based comparators paired with the four CNN input conditions. Their membership is defined by `inputs/FeatureListAnnotated.csv` and restricted to features retained by notebook 00 quality control. Their labels identify the comparison being made and do not assert that every constituent feature is exclusively an image or mask feature. The all-feature RF uses the complete 277-feature correlation-reduced matrix.

The 277-feature all-feature matrix is batch-corrected independently within each complete experimental batch using only morphology predictors; the four subset comparators use the post-QC raw matrix, matching the source workflow. The batch correction is an unsupervised, whole-batch normalization step: evaluation assumes that the unlabeled distribution of a new batch is available for normalization and that batch-level location/scale shifts are technical. HVG selection and FUCCI preprocessing are also global, exactly as in the source workflow. Gene standardization and linear residualization on `sin(FUCCI angle)` and `cos(FUCCI angle)` are fitted using each training fold and applied to its held-out batch.

The retained permutation analysis is a single sanity-check permutation. It pools the 2,000 permuted-gene correlations and records their 95th percentile, historically about Pearson `r = 0.03`. The publication effect threshold is `r > 0.1`; the permutation result is not presented as formal permutation-based error control. Fisher-combined p-values and BH values are retained to reproduce the source workflow.

SHAP uses held-out cells from each leave-one-batch-out fold, but its random forests are 200-tree surrogates for computational tractability. These SHAP surrogates differ from the 1,000-tree random forests used for evaluated prediction performance; they retain the same depth, feature-sampling, and random-state settings. SHAP is calculated for the historical `FDR < 0.01` set, while publication effect-size filtering is applied downstream where specified.


In [ ]:
from pathlib import Path
import csv
import json
import os

import anndata as ad
import h5py
import numpy as np
import pandas as pd
import scanpy as sc
from scipy.stats import combine_pvalues, pearsonr
from sklearn.decomposition import PCA
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import StandardScaler
from statsmodels.stats.multitest import multipletests
from tqdm.auto import tqdm

try:
    import shap
except ImportError as exc:
    raise ImportError("SHAP is required for this publication training workflow.") from exc

ROOT = Path.cwd().resolve()
INPUT_DIR = ROOT / "inputs"
SOURCE_OUTPUT_DIR = ROOT / "outputs"
OUTPUT_DIR = SOURCE_OUTPUT_DIR / "rerun"
CELL_DS_PATH = Path(
    os.environ.get("FUCCI_H5_PATH", INPUT_DIR / "fucci_final_nucmask.h5")
).expanduser()
RAW_FEATURES_PATH = SOURCE_OUTPUT_DIR / "features_combined_raw.csv"
FEATURES_PATH = SOURCE_OUTPUT_DIR / "features_combined_corrected.csv"
FEATURE_ANNOTATIONS_PATH = INPUT_DIR / "FeatureListAnnotated.csv"
WHITELIST_PATH = INPUT_DIR / "cid_whitelist.csv"
REMOVE_PATH = INPUT_DIR / "cid_nuc_toremove.csv"

for required_path in [
    CELL_DS_PATH,
    RAW_FEATURES_PATH,
    FEATURES_PATH,
    FEATURE_ANNOTATIONS_PATH,
    WHITELIST_PATH,
    REMOVE_PATH,
]:
    if not required_path.is_file():
        raise FileNotFoundError(f"Required publication input is missing: {required_path}")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

RANDOM_STATE = 42
N_HVGS = 2000
EXPECTED_N_CELLS = 5069
EXPECTED_BATCHES = ("JP241", "JP277", "NG015", "NG026", "NG029", "NG049")
EXPECTED_GENE_SET_SIZES = {"Cell Cycle": 91, "Contractility": 59}
EXPECTED_ALL_FEATURES = 277
EXPECTED_RAW_FEATURES = 485
EXPECTED_ANNOTATION_ROWS = 521
EXPECTED_ANNOTATION_ONLY_FEATURES = 36
EXPECTED_HYBRID_FEATURES = 30
EXPECTED_COMPARISON_COUNTS = {
    "nuc_img": 126,
    "nuc_shape": 108,
    "cell_img": 113,
    "cell_shape": 108,
}
EXPECTED_HISTORICAL_FDR_GENES = 403
EXPECTED_DUAL_FILTER_GENES = 187
RF_PARAMS = dict(n_estimators=1000, max_depth=6, n_jobs=-1, random_state=RANDOM_STATE, max_features=10)
SHAP_RF_PARAMS = dict(n_estimators=200, max_depth=6, n_jobs=-1, random_state=RANDOM_STATE, max_features=10)
SHAP_SAMPLE_SIZE = 100
CONDITION_ORDER = ["Unresidualized", "FUCCI residualized"]
NEAR_ZERO_STD = 1e-8
SIGNIFICANCE_FDR = 0.01
PUBLICATION_EFFECT_THRESHOLD = 0.1
N_PERMUTATIONS = 1
PERMUTATION_RANDOM_SEED = 1729
PERMUTATION_THRESHOLD_QUANTILE = 0.95

CONTRACTILITY_GENES = ['1600019K03Rik', 'Acta2', 'Actb', 'Adamts5', 'Adh5', 'Anxa1', 'Anxa2', 'Anxa3', 'Anxa5', 'Bgn', 'Blvrb', 'Cald1', 'Ccne2', 'Cd63', 'Cd9', 'Cdkn1a', 'Cope', 'Cst3', 'Ctgf', 'Cyp51', 'Ddit3', 'Eif4a3', 'Eno1', 'Fbln5', 'Hspa5', 'Id3', 'Ifitm3', 'Igfbp7', 'Itm2b', 'Lgals3', 'Mustn1', 'Myl12a', 'Ndufc2', 'Pcna-ps2', 'Pcolce', 'Pdlim1', 'Pgp', 'Phgdh', 'Prdx1', 'Prmt1', 'Prss23', 'Ptgr1', 'Ptn', 'Ptrh2', 'Rrm2', 'Sdhb', 'Selenok', 'Serpinf1', 'Slc25a5', 'Slpi', 'Steap1', 'Tagln', 'Thbs1', 'Timp1', 'Tpm2', 'Tubb4b', 'Ubb', 'Ubc', 'Uqcrb']
CC_GENES = ['1700061I17Rik', 'Amotl2', 'Arhgap11a', 'Aspm', 'Atad5', 'Aurka', 'Birc5', 'Bub1', 'Ccdc77', 'Ccna2', 'Ccnb1', 'Ccnf', 'Cdc20', 'Cdca8', 'Cdk1', 'Cenpe', 'Cenpf', 'Cep55', 'Cfap69', 'Ckap2', 'Ckap2l', 'Cks2', 'Col11a1', 'Cryab', 'Cxcl5', 'Cyr61', 'Dlgap1', 'Gas2l3', 'Glipr1', 'Gm10701', 'Gm10801', 'Gm17501', 'Gm26870', 'Gm43154', 'Gm5886', 'H1f0', 'Hist1h1b', 'Hist1h1d', 'Hist1h1e', 'Hist1h2ab', 'Hist1h2ae', 'Hist1h2ai', 'Hist1h2ak', 'Hist1h2an', 'Hist1h3c', 'Hist1h4c', 'Hist1h4d', 'Hist2h2ac', 'Hmgn5', 'Hmmr', 'Il1rl1', 'Immp2l', 'Incenp', 'Kif11', 'Kif18a', 'Kif20b', 'Knl1', 'Kpna2', 'Lockd', 'Mad1l1', 'Mettl15', 'Mis18bp1', 'Mki67', 'Negr1', 'Nusap1', 'Nxt1', 'Parpbp', 'Prc1', 'Ptgs2', 'Rad51ap1', 'Rora', 'Rxfp1', 'S100a7a', 'Sapcd2', 'Sdhaf4', 'Serpinb9b', 'Sgo1', 'Sgo2a', 'Slfn9', 'Smc4', 'Sorbs1', 'Spc25', 'Spdl1', 'Sulf2', 'Top2a', 'Tpx2', 'Ube2c', 'Usp14', 'Vgll3', 'Wtap', 'Zfpm2']


## Fold-aware training helpers


In [ ]:
# Embedded HDF5-to-AnnData loader.

def get_exp_data(gene_exp: bytes) -> dict:
    decode_data = gene_exp.decode("utf-8")
    decode_data = decode_data.replace("\'","\"")
    exp_dict = json.loads(decode_data)
    return exp_dict


def expdf_from_h5(h5_path):
    exp_list = []
    with h5py.File(h5_path, 'r') as dataset:
        cell_ids = list(dataset.keys())
        for cell_id in cell_ids:
            cell_ds = dataset[cell_id]
            if "exp_data" in cell_ds and cell_ds.attrs['num_cells'] == 1:
                try:
                    exp_data = cell_ds['exp_data/exp_data'][()]
                except:
                    print("no exp dataset:", cell_id)
                else:
                    exp_dict = get_exp_data(exp_data)
                    exp_series = pd.Series(exp_dict)
                    exp_series.name = cell_id
                    exp_list.append(exp_series)

    exp_df = pd.DataFrame(exp_list)
    return exp_df


def metadf_from_h5(h5_path):
    meta_list = []
    with h5py.File(h5_path, 'r') as dataset:
        cell_ids = list(dataset.keys())
        for cell_id in cell_ids:
            cell_ds = dataset[cell_id]

            if "exp_data/exp_data" in cell_ds and cell_ds.attrs['num_cells'] == 1:
                attr_list = [('cell_id', cell_id)]

                for attr_name in cell_ds.attrs.keys():
                    attr_value = cell_ds.attrs[attr_name] 
                    attr_list.append((attr_name, attr_value))

                for attr_name in cell_ds['exp_data'].attrs.keys():
                    attr_value = cell_ds['exp_data'].attrs[attr_name] 
                    attr_list.append((attr_name, attr_value))

                attr_dict = {at_k: at_v for at_k, at_v in attr_list}
                meta_list.append(pd.Series(attr_dict))
    meta_df = pd.DataFrame(meta_list) 
    meta_df.set_index('cell_id', inplace=True)
    return meta_df


def annd_from_h5(h5_path):
    exp_df = expdf_from_h5(h5_path)
    meta_df = metadf_from_h5(h5_path)
    exp_df = exp_df.fillna(0)
    adata = ad.AnnData(exp_df, exp_df.index.to_frame(), exp_df.columns.to_frame())
    adata.obs = meta_df
    print(adata)
    return adata


def read_cell_list(path):
    with open(path, newline="") as handle:
        rows = list(csv.reader(handle))
    if not rows:
        return []
    return [str(x) for row in rows for x in row if str(x).strip()]


def to_dense_df(adata_view):
    matrix = adata_view.X
    if hasattr(matrix, "toarray"):
        matrix = matrix.toarray()
    return pd.DataFrame(matrix, index=adata_view.obs_names.astype(str), columns=adata_view.var_names.astype(str))


def compute_fucci_angle(obs, channels=("488_int_mean", "561_int_mean")):
    missing = [c for c in channels if c not in obs.columns]
    if missing:
        raise KeyError(f"Missing FUCCI channels: {missing}")
    fucci = obs.loc[:, list(channels)].astype(float).dropna()
    fucci_log = np.log1p(fucci)
    fucci_scaled = StandardScaler().fit_transform(fucci_log)
    pcs = PCA(n_components=2, random_state=RANDOM_STATE).fit_transform(fucci_scaled)
    angle = np.mod(np.arctan2(pcs[:, 1], pcs[:, 0]), 2 * np.pi)
    return pd.DataFrame({
        "cell_id": fucci.index.astype(str),
        channels[0]: fucci[channels[0]].values,
        channels[1]: fucci[channels[1]].values,
        "fucci_angular_score": angle,
    }).set_index("cell_id")


def safe_pearson(x, y):
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)
    valid = np.isfinite(x) & np.isfinite(y)
    if valid.sum() < 3 or np.std(x[valid]) == 0 or np.std(y[valid]) == 0:
        return np.nan, np.nan
    return pearsonr(x[valid], y[valid])


def fdr_bh(pvalues):
    pvalues = pd.Series(pvalues, dtype=float).fillna(1.0)
    return multipletests(pvalues, method="fdr_bh")[1]


def build_exp_dict(cell_metadata):
    return cell_metadata.groupby("exp_id").apply(lambda x: x.index.astype(str).tolist()).to_dict()


def iter_leave_one_batch(exp_dict):
    exp_ids = list(exp_dict.keys())
    for test_batch in exp_ids:
        train_batches = [batch for batch in exp_ids if batch != test_batch]
        test_cells = set(map(str, exp_dict[test_batch]))
        train_cells = set(str(cell) for batch in train_batches for cell in exp_dict[batch])
        yield test_batch, train_cells, test_cells


def train_leave_one_batch_rf(X, Y, exp_dict, target_genes, rf_params, collect_predictions=True):
    gene_rows = []
    batch_rows = []
    pred_rows = []

    for gene in tqdm(target_genes, desc="RF genes"):
        if gene not in Y.columns or not pd.api.types.is_numeric_dtype(Y[gene]):
            continue

        y = Y[gene]
        fold_rs = []
        fold_ps = []
        train_rs = []

        for test_batch, train_cells, test_cells in iter_leave_one_batch(exp_dict):
            valid = y.notna()
            idx_train = X.index[valid & X.index.isin(train_cells)]
            idx_test = X.index[valid & X.index.isin(test_cells)]
            if len(idx_train) < 3 or len(idx_test) < 3:
                continue

            X_train = X.loc[idx_train]
            X_test = X.loc[idx_test]
            y_train = y.loc[idx_train]
            y_test = y.loc[idx_test]

            rf = RandomForestRegressor(**rf_params)
            rf.fit(X_train, y_train)
            pred_train = rf.predict(X_train)
            pred_test = rf.predict(X_test)

            r, p = safe_pearson(y_test.values, pred_test)
            r_train, _ = safe_pearson(y_train.values, pred_train)
            fold_rs.append(r)
            fold_ps.append(p)
            train_rs.append(r_train)

            batch_rows.append({"gene": gene, "test_batch": test_batch, "pearson_r": r, "p_value": p, "n_test": len(idx_test)})
            if collect_predictions:
                pred_rows.extend({
                    "cell_id": cell_id,
                    "gene": gene,
                    "actual": actual,
                    "predicted": predicted,
                    "test_batch": test_batch,
                } for cell_id, actual, predicted in zip(idx_test.astype(str), y_test.values, pred_test))

        if fold_rs:
            finite_ps = [p for p in fold_ps if np.isfinite(p)]
            combined_p = combine_pvalues(finite_ps, method="fisher")[1] if finite_ps else np.nan
            gene_rows.append({
                "gene": gene,
                "avg_Pearson": np.nanmean(fold_rs),
                "combined_p": combined_p,
                "train_avg_Pearson": np.nanmean(train_rs),
                "n_folds": len(fold_rs),
            })

    metrics = pd.DataFrame(gene_rows).sort_values("avg_Pearson", ascending=False)
    if not metrics.empty:
        metrics["fdr"] = fdr_bh(metrics["combined_p"])
    return metrics, pd.DataFrame(batch_rows), pd.DataFrame(pred_rows)


def shap_for_gene(X, y, exp_dict, target_name, rf_params, sample_size):
    if shap is None:
        raise ImportError("Install shap to run SHAP extraction")
    fold_values = []
    for test_batch, train_cells, test_cells in tqdm(list(iter_leave_one_batch(exp_dict)), desc=f"SHAP {target_name}"):
        idx_train = X.index[X.index.isin(train_cells)].intersection(y.dropna().index)
        idx_test = X.index[X.index.isin(test_cells)].intersection(y.dropna().index)
        if len(idx_train) < 3 or len(idx_test) < 3:
            continue
        rf = RandomForestRegressor(**rf_params)
        rf.fit(X.loc[idx_train], y.loc[idx_train])
        X_sample = X.loc[idx_test].sample(min(sample_size, len(idx_test)), random_state=RANDOM_STATE)
        explainer = shap.TreeExplainer(rf)
        values = explainer.shap_values(X_sample)
        fold_values.append(np.mean(np.abs(values), axis=0))
    return pd.DataFrame({target_name: np.mean(fold_values, axis=0)}, index=X.columns)


def shap_for_genes(X, Y, exp_dict, genes, rf_params, sample_size):
    rows = []
    for gene in tqdm(genes, desc="SHAP significant genes"):
        if gene not in Y.columns:
            continue
        gene_shap = shap_for_gene(
            X=X,
            y=Y[gene],
            exp_dict=exp_dict,
            target_name="mean_abs_shap",
            rf_params=rf_params,
            sample_size=sample_size,
        )
        gene_shap = gene_shap.reset_index().rename(columns={"index": "feature"})
        gene_shap["gene"] = gene
        rows.append(gene_shap[["gene", "feature", "mean_abs_shap"]])
    if not rows:
        return pd.DataFrame(columns=["gene", "feature", "mean_abs_shap"])
    return pd.concat(rows, ignore_index=True)


## Global preprocessing and the 5,069-cell intersection

The whitelist/removal filters and intersection of combined morphology features, 2,000-HVG expression, cell metadata, and non-missing FUCCI angles retain the six publication batches and 5,069 cells.


In [ ]:
adata_org = annd_from_h5(CELL_DS_PATH)
adata = adata_org.copy()
adata.obs_names = adata.obs_names.astype(str)

adata.layers["raw_counts"] = adata.X.copy()
filter_mask, _ = sc.pp.filter_cells(adata, min_genes=500, inplace=False)
adata = adata[filter_mask].copy()
sc.pp.filter_genes(adata, min_cells=5)
sc.pp.normalize_total(adata)
sc.pp.log1p(adata)
sc.pp.highly_variable_genes(adata, n_top_genes=N_HVGS, flavor="seurat", batch_key="exp_id")

df_hvgs = to_dense_df(adata[:, adata.var["highly_variable"]])

to_keep = set(read_cell_list(WHITELIST_PATH))
to_remove = set(read_cell_list(REMOVE_PATH))
keep_index = [idx for idx in df_hvgs.index.astype(str) if idx in to_keep and idx not in to_remove]
df_hvgs = df_hvgs.loc[keep_index]

df_features = pd.read_csv(FEATURES_PATH, index_col=0)
df_features_raw = pd.read_csv(RAW_FEATURES_PATH, index_col=0)
for feature_frame in (df_features, df_features_raw):
    feature_frame.index = feature_frame.index.astype(str)
    if "exp_id" in feature_frame.columns:
        feature_frame.drop(columns=["exp_id"], inplace=True)
    feature_frame[:] = feature_frame.apply(pd.to_numeric, errors="coerce")

cell_metadata = adata.obs.copy()
cell_metadata.index = cell_metadata.index.astype(str)
fucci_angles = compute_fucci_angle(cell_metadata)

common = (
    df_features.index.intersection(df_features_raw.index)
    .intersection(df_hvgs.index)
    .intersection(cell_metadata.index)
    .intersection(fucci_angles.index)
)
df_features = df_features.loc[common].dropna(axis=1, how="all")
df_features_raw = df_features_raw.loc[common].dropna(axis=1, how="all")
df_hvgs = df_hvgs.loc[common]
cell_metadata = cell_metadata.loc[common]
fucci_angles = fucci_angles.loc[common]

required_meta_cols = [c for c in ["exp_id", "488_int_mean", "561_int_mean"] if c in cell_metadata.columns]
cell_metadata_out = cell_metadata[required_meta_cols].copy()
cell_metadata_out.index.name = "cell_id"
cell_metadata_out = cell_metadata_out.join(fucci_angles[["fucci_angular_score"]], how="left")

exp_dict = build_exp_dict(cell_metadata_out)

if len(common) != EXPECTED_N_CELLS:
    raise ValueError(f"Expected {EXPECTED_N_CELLS} intersected cells, found {len(common)}")
if tuple(exp_dict) != EXPECTED_BATCHES:
    raise ValueError(f"Expected batches {EXPECTED_BATCHES}, found {tuple(exp_dict)}")
if df_hvgs.shape[1] != N_HVGS:
    raise ValueError(f"Expected {N_HVGS} HVGs, found {df_hvgs.shape[1]}")

cell_metadata_out.to_csv(OUTPUT_DIR / "cell_metadata.csv")

print("correlation-reduced features", df_features.shape)
print("raw modality features", df_features_raw.shape)
print("HVG expression", df_hvgs.shape)
print("batches", list(exp_dict.keys()))

target_genes = list(df_hvgs.columns)

## All-feature RF for all HVGs


In [ ]:
target_genes = list(df_hvgs.columns)
rf_gene_metrics, rf_gene_batch_metrics, rf_test_predictions = train_leave_one_batch_rf(
    df_features, df_hvgs, exp_dict, target_genes, RF_PARAMS
)

publication_gene_mask = (
    (rf_gene_metrics["fdr"] < SIGNIFICANCE_FDR)
    & (rf_gene_metrics["avg_Pearson"] > PUBLICATION_EFFECT_THRESHOLD)
)
print(
    f"Publication dual-filter genes: {publication_gene_mask.sum()} "
    f"(FDR < {SIGNIFICANCE_FDR} and r > {PUBLICATION_EFFECT_THRESHOLD})"
)

rf_gene_metrics.to_csv(OUTPUT_DIR / "rf_gene_metrics.csv", index=False)
rf_gene_batch_metrics.to_csv(OUTPUT_DIR / "rf_gene_batch_metrics.csv", index=False)
rf_test_predictions.to_csv(OUTPUT_DIR / "rf_test_predictions.csv.gz", index=False)


## Single-permutation sanity check

This reproduces the source workflow's single shuffled-expression run. The 95th percentile
of the 2,000 pooled null-gene correlations is a descriptive chance-level reference, not a
formal permutation-based FDR. The publication uses the more conservative effect threshold
`r > 0.1` where effect-size filtering is specified.


In [ ]:
rng = np.random.default_rng(PERMUTATION_RANDOM_SEED)
permutation_metric_tables = []

for permutation_id in range(N_PERMUTATIONS):
    print(f"Permutation sanity-check run {permutation_id + 1}/{N_PERMUTATIONS}")
    df_hvgs_permuted = df_hvgs.copy()
    for gene in target_genes:
        values = df_hvgs_permuted[gene].to_numpy(copy=True)
        valid = np.isfinite(values)
        shuffled = values.copy()
        shuffled[valid] = rng.permutation(values[valid])
        df_hvgs_permuted[gene] = shuffled

    perm_metrics, perm_batch_metrics, _ = train_leave_one_batch_rf(
        df_features,
        df_hvgs_permuted,
        exp_dict,
        target_genes,
        RF_PARAMS,
        collect_predictions=False,
    )
    perm_metrics["permutation_id"] = permutation_id
    perm_batch_metrics["permutation_id"] = permutation_id
    permutation_metric_tables.append(perm_metrics)
    perm_batch_metrics.to_csv(
        OUTPUT_DIR / f"rf_permutation_null_batch_metrics_perm{permutation_id:03d}.csv",
        index=False,
    )

rf_permutation_null_metrics = pd.concat(permutation_metric_tables, ignore_index=True)
rf_permutation_null_metrics.to_csv(
    OUTPUT_DIR / "rf_permutation_null_gene_metrics.csv", index=False
)

null_threshold = rf_permutation_null_metrics["avg_Pearson"].quantile(
    PERMUTATION_THRESHOLD_QUANTILE
)
null_summary = pd.DataFrame([{
    "n_permutations": N_PERMUTATIONS,
    "threshold_quantile": PERMUTATION_THRESHOLD_QUANTILE,
    "avg_Pearson_null_threshold": null_threshold,
    "publication_effect_threshold": PUBLICATION_EFFECT_THRESHOLD,
    "null_mean_avg_Pearson": rf_permutation_null_metrics["avg_Pearson"].mean(),
    "null_sd_avg_Pearson": rf_permutation_null_metrics["avg_Pearson"].std(),
}])
null_summary.to_csv(OUTPUT_DIR / "rf_permutation_null_threshold.csv", index=False)

null_by_gene = rf_permutation_null_metrics.groupby("gene")["avg_Pearson"].agg(
    null_mean="mean",
    null_sd="std",
    null_max="max",
).reset_index()
rf_gene_metrics = rf_gene_metrics.merge(null_by_gene, on="gene", how="left")
rf_gene_metrics["above_global_permutation_threshold"] = (
    rf_gene_metrics["avg_Pearson"] > null_threshold
)

# Diagnostic only: one pooled permutation does not provide gene-specific inference.
null_values = rf_permutation_null_metrics["avg_Pearson"].dropna().to_numpy()
rf_gene_metrics["empirical_permutation_p"] = rf_gene_metrics["avg_Pearson"].apply(
    lambda observed: (1 + np.sum(null_values >= observed)) / (1 + len(null_values))
)
rf_gene_metrics["empirical_permutation_fdr"] = fdr_bh(
    rf_gene_metrics["empirical_permutation_p"]
)
rf_gene_metrics.to_csv(OUTPUT_DIR / "rf_gene_metrics.csv", index=False)

print(null_summary.to_string(index=False))


## Feature RFs paired with CNN input conditions

The publication compares four annotated feature subsets with CNNs trained from Hoechst,
nucleus-segmentation, brightfield, and cell-segmentation inputs. Membership is read from
`inputs/FeatureListAnnotated.csv` and intersected with notebook 00's post-QC raw matrix. These are
historical comparison labels and do not imply that every feature in a set is exclusively
image- or mask-derived. The post-QC set sizes are 126, 108, 113, and 108 features,
respectively. Each comparator uses the all-feature model's 1,000-tree leave-one-batch-out RF
parameters and Fisher/BH aggregation.


In [ ]:
comparison_specs = {
    "nuc_img": ("nucleus", "img", "nuc_img_feature"),
    "nuc_shape": ("nucleus", "mask", "nuc_shape_feature"),
    "cell_img": ("bf", "img", "cell_img_feature"),
    "cell_shape": ("bf", "mask", "cell_shape_feature"),
}


def comparison_feature_groups(annotation_path, columns):
    annotations = pd.read_csv(annotation_path, dtype=str)
    required_columns = ["Name", "Channel", "Type"]
    if list(annotations.columns) != required_columns:
        raise ValueError(
            f"Feature annotations must have columns {required_columns}, "
            f"found {list(annotations.columns)}"
        )
    if len(annotations) != EXPECTED_ANNOTATION_ROWS:
        raise ValueError(
            f"Expected {EXPECTED_ANNOTATION_ROWS} feature annotations, "
            f"found {len(annotations)}"
        )
    if annotations[required_columns].isna().any().any():
        raise ValueError("Feature annotations contain missing values")
    if annotations["Name"].duplicated().any():
        duplicates = annotations.loc[annotations["Name"].duplicated(), "Name"].tolist()
        raise ValueError(f"Duplicate feature annotations: {duplicates[:10]}")

    expected_annotation_pairs = {
        ("nucleus", "img"),
        ("nucleus", "mask"),
        ("bf", "img"),
        ("bf", "mask"),
        ("hybrid", "hybrid"),
    }
    observed_annotation_pairs = set(
        annotations[["Channel", "Type"]].itertuples(index=False, name=None)
    )
    if observed_annotation_pairs != expected_annotation_pairs:
        raise ValueError(
            f"Unexpected feature annotation categories: {observed_annotation_pairs}"
        )

    raw_columns = pd.Index(columns).astype(str)
    if raw_columns.has_duplicates:
        raise ValueError("Raw feature matrix contains duplicate columns")
    if len(raw_columns) != EXPECTED_RAW_FEATURES:
        raise ValueError(
            f"Expected {EXPECTED_RAW_FEATURES} post-QC raw features, "
            f"found {len(raw_columns)}"
        )
    raw_feature_set = set(raw_columns)
    annotation_feature_set = set(annotations["Name"])
    unannotated = raw_feature_set - annotation_feature_set
    if unannotated:
        raise ValueError(f"Post-QC raw features lack annotations: {sorted(unannotated)[:10]}")

    annotation_only = annotations.loc[
        ~annotations["Name"].isin(raw_feature_set)
    ]
    annotation_only_pairs = set(
        annotation_only[["Channel", "Type"]].itertuples(index=False, name=None)
    )
    if (
        len(annotation_only) != EXPECTED_ANNOTATION_ONLY_FEATURES
        or annotation_only_pairs != {("bf", "img")}
        or not annotation_only["Name"].str.startswith("cell_boundary_").all()
    ):
        raise ValueError(
            "Expected exactly 36 annotation-only bf/img cell_boundary features"
        )

    groups = {}
    for output_stem, (channel, feature_type, _) in comparison_specs.items():
        annotated_names = annotations.loc[
            (annotations["Channel"] == channel)
            & (annotations["Type"] == feature_type),
            "Name",
        ]
        groups[output_stem] = [
            name for name in annotated_names if name in raw_feature_set
        ]

    observed_counts = {name: len(features) for name, features in groups.items()}
    if observed_counts != EXPECTED_COMPARISON_COUNTS:
        raise ValueError(
            f"Unexpected comparator feature counts: {observed_counts}; "
            f"expected {EXPECTED_COMPARISON_COUNTS}"
        )
    comparator_features = set().union(*(set(features) for features in groups.values()))
    if len(comparator_features) != sum(observed_counts.values()):
        raise ValueError("Annotated comparator feature groups overlap")

    hybrid_features = set(annotations.loc[
        (annotations["Channel"] == "hybrid")
        & (annotations["Type"] == "hybrid"),
        "Name",
    ]) & raw_feature_set
    if len(hybrid_features) != EXPECTED_HYBRID_FEATURES:
        raise ValueError(
            f"Expected {EXPECTED_HYBRID_FEATURES} post-QC hybrid features, "
            f"found {len(hybrid_features)}"
        )
    if raw_feature_set - comparator_features != hybrid_features:
        raise ValueError("Only annotated hybrid features may be excluded from comparators")
    return groups, hybrid_features


def train_comparator_rf(X, Y, exp_dict, target_genes, rf_params):
    rows = []
    for gene in tqdm(target_genes, desc=f"RF comparator ({X.shape[1]} features)"):
        if gene not in Y.columns or not pd.api.types.is_numeric_dtype(Y[gene]):
            continue
        y = Y[gene]
        fold_rs = []
        fold_ps = []
        for _, train_cells, test_cells in iter_leave_one_batch(exp_dict):
            valid = y.notna()
            idx_train = X.index[valid & X.index.isin(train_cells)]
            idx_test = X.index[valid & X.index.isin(test_cells)]
            if len(idx_train) < 3 or len(idx_test) < 3:
                continue
            model = RandomForestRegressor(**rf_params)
            model.fit(X.loc[idx_train], y.loc[idx_train])
            prediction = model.predict(X.loc[idx_test])
            pearson_r, p_value = safe_pearson(y.loc[idx_test].values, prediction)
            fold_rs.append(pearson_r)
            fold_ps.append(p_value)
        finite_ps = [p for p in fold_ps if np.isfinite(p)]
        if fold_rs:
            rows.append({
                "gene": gene,
                "avg_Pearson": np.nanmean(fold_rs),
                "combined_p": combine_pvalues(finite_ps, method="fisher")[1]
                if finite_ps else np.nan,
            })
    metrics = pd.DataFrame(rows).sort_values("avg_Pearson", ascending=False)
    metrics["avg_Pearson_fdr"] = fdr_bh(metrics["combined_p"])
    return metrics


def write_comparator_metrics(metrics, metric_key, path):
    output = metrics.set_index("gene")[["avg_Pearson", "avg_Pearson_fdr"]].copy()
    output.index.name = None
    output.columns = pd.MultiIndex.from_tuples(
        [
            ("avg_Pearson", metric_key),
            ("avg_Pearson_fdr", metric_key),
        ],
        names=[None, "channel"],
    )
    output.to_csv(path)

    roundtrip = pd.read_csv(path, header=[0, 1], index_col=0)
    if roundtrip.shape != (N_HVGS, 2) or "gene" in roundtrip.index:
        raise ValueError(f"Invalid comparator RF CSV serialization: {path}")
    if not roundtrip.index.is_unique:
        raise ValueError(f"Comparator RF CSV has duplicate genes: {path}")


feature_groups, hybrid_features = comparison_feature_groups(
    FEATURE_ANNOTATIONS_PATH, df_features_raw.columns
)
observed_comparison_counts = {
    name: len(features) for name, features in feature_groups.items()
}
print(pd.Series(observed_comparison_counts, name="n_features"))

comparison_results = {}
for output_stem, (_, _, metric_key) in comparison_specs.items():
    columns = feature_groups[output_stem]
    metrics = train_comparator_rf(
        df_features_raw.loc[:, columns],
        df_hvgs,
        exp_dict,
        target_genes,
        RF_PARAMS,
    )
    write_comparator_metrics(
        metrics,
        metric_key,
        OUTPUT_DIR / f"Result_RF_{output_stem}fdr.csv",
    )
    comparison_results[output_stem] = metrics


## Per-gene FUCCI-residual comparison (91 cell-cycle and 59 contractility genes)


In [14]:
gene_lookup = {gene.upper(): gene for gene in df_hvgs.columns.astype(str)}

def genes_available_in_expression(genes):
    available = []
    seen = set()
    for gene in genes:
        matched_gene = gene if gene in df_hvgs.columns else gene_lookup.get(str(gene).upper())
        if matched_gene is not None and matched_gene not in seen:
            available.append(matched_gene)
            seen.add(matched_gene)
    return available

individual_gene_sets = {
    "Cell Cycle": genes_available_in_expression(CC_GENES),
    "Contractility": genes_available_in_expression(CONTRACTILITY_GENES),
}
observed_gene_set_sizes = {module: len(genes) for module, genes in individual_gene_sets.items()}
if observed_gene_set_sizes != EXPECTED_GENE_SET_SIZES:
    raise ValueError(
        f"Expected per-gene sets {EXPECTED_GENE_SET_SIZES}, found {observed_gene_set_sizes}"
    )
print(observed_gene_set_sizes)

fucci_covariates = pd.DataFrame({
    "sin_fucci": np.sin(fucci_angles.loc[df_features.index, "fucci_angular_score"]),
    "cos_fucci": np.cos(fucci_angles.loc[df_features.index, "fucci_angular_score"]),
}, index=df_features.index)

individual_gene_fold_rows = []
folds = list(iter_leave_one_batch(exp_dict))

for module, genes in individual_gene_sets.items():
    for gene in tqdm(genes, desc=f"Individual genes {module}"):
        y = df_hvgs[gene].astype(float)
        for test_batch, train_cells, test_cells in folds:
            idx_train = df_features.index[df_features.index.isin(train_cells)]
            idx_test = df_features.index[df_features.index.isin(test_cells)]
            idx_train = idx_train.intersection(y.dropna().index).intersection(fucci_covariates.dropna().index)
            idx_test = idx_test.intersection(y.dropna().index).intersection(fucci_covariates.dropna().index)
            if len(idx_train) < 3 or len(idx_test) < 3:
                continue

            y_train = y.loc[idx_train]
            y_test = y.loc[idx_test]
            train_std = y_train.std()
            if not np.isfinite(train_std) or train_std <= NEAR_ZERO_STD:
                continue

            train_mean = y_train.mean()
            y_train_z = (y_train - train_mean) / train_std
            y_test_z = (y_test - train_mean) / train_std

            fold_targets = {"Unresidualized": (y_train_z.values, y_test_z.values)}
            resid_model = LinearRegression().fit(fucci_covariates.loc[idx_train], y_train_z)
            fold_targets["FUCCI residualized"] = (
                y_train_z.values - resid_model.predict(fucci_covariates.loc[idx_train]),
                y_test_z.values - resid_model.predict(fucci_covariates.loc[idx_test]),
            )

            for condition, (target_train, target_test) in fold_targets.items():
                if np.nanstd(target_train) <= NEAR_ZERO_STD or np.nanstd(target_test) <= NEAR_ZERO_STD:
                    continue
                rf = RandomForestRegressor(**RF_PARAMS)
                rf.fit(df_features.loc[idx_train], target_train)
                pred_test = rf.predict(df_features.loc[idx_test])
                r, _ = safe_pearson(target_test, pred_test)
                if not np.isfinite(r):
                    continue
                individual_gene_fold_rows.append({
                    "gene": gene,
                    "module": module,
                    "fold": test_batch,
                    "condition": condition,
                    "pearson_r": r,
                    "n_test_cells": len(idx_test),
                })

individual_gene_fold_results = pd.DataFrame(individual_gene_fold_rows)
if individual_gene_fold_results.empty:
    raise ValueError("No valid individual-gene prediction results were generated.")

individual_gene_fold_results.to_csv(
    OUTPUT_DIR / "individual_gene_fucci_residual_fold_metrics.csv",
    index=False,
)

individual_gene_mean_results = (
    individual_gene_fold_results
    .groupby(["gene", "module", "condition"], as_index=False)
    .agg(pearson_r=("pearson_r", "mean"), n_folds=("pearson_r", "size"))
)

paired_gene_results = (
    individual_gene_mean_results
    .pivot(index=["gene", "module"], columns="condition", values="pearson_r")
    .dropna(subset=CONDITION_ORDER)
    .reset_index()
)
paired_gene_results["delta_r"] = paired_gene_results["FUCCI residualized"] - paired_gene_results["Unresidualized"]
individual_gene_mean_results = individual_gene_mean_results.merge(
    paired_gene_results[["gene", "module", "delta_r"]],
    on=["gene", "module"],
    how="left",
)
individual_gene_mean_results.to_csv(
    OUTPUT_DIR / "individual_gene_fucci_residual_gene_mean_metrics.csv",
    index=False,
)

summary_rows = []
for module, module_df in paired_gene_results.groupby("module"):
    unres = module_df["Unresidualized"]
    resid = module_df["FUCCI residualized"]
    summary_rows.append({
        "module": module,
        "n_retained_genes": len(module_df),
        "median_unresidualized_pearson_r": unres.median(),
        "iqr_unresidualized_pearson_r": unres.quantile(0.75) - unres.quantile(0.25),
        "median_fucci_residualized_pearson_r": resid.median(),
        "iqr_fucci_residualized_pearson_r": resid.quantile(0.75) - resid.quantile(0.25),
        "median_delta_r": module_df["delta_r"].median(),
    })
individual_gene_summary = pd.DataFrame(summary_rows)
individual_gene_summary.to_csv(
    OUTPUT_DIR / "individual_gene_fucci_residual_summary.csv",
    index=False,
)


{'Cell Cycle': 91, 'Contractility': 59}


Individual genes Contractility: 100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 59/59 [15:13<00:00, 15.49s/it]


## Held-out SHAP surrogates for the historical FDR-selected genes

Calculates SHAP for significant genes.

In [ ]:
significant_genes = rf_gene_metrics.loc[
    rf_gene_metrics["fdr"] < SIGNIFICANCE_FDR, "gene"
].tolist()
print(f"Computing held-out SHAP surrogates for {len(significant_genes)} historical FDR genes")

shap_significant_genes = shap_for_genes(
    df_features, df_hvgs, exp_dict, significant_genes, SHAP_RF_PARAMS, SHAP_SAMPLE_SIZE
)
shap_significant_genes.to_csv(
    OUTPUT_DIR / "shap_significant_genes.csv.gz", index=False
)


## Validate the staged rerun

This cell validates the complete staged run. It does not copy or replace publication files.
Follow `README.md` only after this cell succeeds.


In [ ]:
replacement_files = [
    "cell_metadata.csv",
    "rf_gene_metrics.csv",
    "rf_test_predictions.csv.gz",
    "Result_RF_nuc_imgfdr.csv",
    "Result_RF_nuc_shapefdr.csv",
    "Result_RF_cell_imgfdr.csv",
    "Result_RF_cell_shapefdr.csv",
    "individual_gene_fucci_residual_fold_metrics.csv",
    "individual_gene_fucci_residual_gene_mean_metrics.csv",
    "individual_gene_fucci_residual_summary.csv",
    "shap_significant_genes.csv.gz",
]
support_files = [
    "rf_gene_batch_metrics.csv",
    "rf_permutation_null_gene_metrics.csv",
    "rf_permutation_null_threshold.csv",
    "rf_permutation_null_batch_metrics_perm000.csv",
]
missing_outputs = [
    name for name in [*replacement_files, *support_files]
    if not (OUTPUT_DIR / name).is_file()
]
if missing_outputs:
    raise FileNotFoundError(f"Missing staged rerun outputs: {missing_outputs}")

if len(cell_metadata_out) != EXPECTED_N_CELLS or not cell_metadata_out.index.is_unique:
    raise ValueError("Cell metadata failed the expected 5,069-cell uniqueness check")
if len(rf_gene_metrics) != N_HVGS or not rf_gene_metrics["gene"].is_unique:
    raise ValueError("RF gene metrics failed the expected 2,000-gene uniqueness check")
if set(rf_gene_metrics["n_folds"]) != {len(EXPECTED_BATCHES)}:
    raise ValueError("Not every all-feature RF gene has six held-out folds")
if len(rf_gene_batch_metrics) != N_HVGS * len(EXPECTED_BATCHES):
    raise ValueError("RF batch metrics do not contain 2,000 genes x six folds")
if len(rf_test_predictions) != N_HVGS * EXPECTED_N_CELLS:
    raise ValueError("RF predictions do not contain 2,000 genes x 5,069 held-out cells")

if observed_comparison_counts != EXPECTED_COMPARISON_COUNTS:
    raise ValueError("Comparator feature counts changed after training")
if len(hybrid_features) != EXPECTED_HYBRID_FEATURES:
    raise ValueError("Post-QC hybrid feature count changed after training")

for output_stem, (_, _, metric_key) in comparison_specs.items():
    path = OUTPUT_DIR / f"Result_RF_{output_stem}fdr.csv"
    frame = pd.read_csv(path, header=[0, 1], index_col=0)
    required = {
        ("avg_Pearson", metric_key),
        ("avg_Pearson_fdr", metric_key),
    }
    if frame.shape != (N_HVGS, 2) or set(frame.columns) != required:
        raise ValueError(f"Comparator schema validation failed: {path}")
    if not frame.index.is_unique or "gene" in frame.index:
        raise ValueError(f"Comparator gene-index validation failed: {path}")

expected_individual_rows = sum(EXPECTED_GENE_SET_SIZES.values()) * len(EXPECTED_BATCHES) * 2
if len(individual_gene_fold_results) != expected_individual_rows:
    raise ValueError(
        f"Expected {expected_individual_rows} individual-gene fold rows, "
        f"found {len(individual_gene_fold_results)}"
    )
if len(individual_gene_mean_results) != sum(EXPECTED_GENE_SET_SIZES.values()) * 2:
    raise ValueError("Individual-gene mean metrics have an unexpected row count")
if set(individual_gene_summary["module"]) != set(EXPECTED_GENE_SET_SIZES):
    raise ValueError("Individual-gene summary has unexpected modules")

if df_features.shape[1] != EXPECTED_ALL_FEATURES:
    raise ValueError(
        f"Expected {EXPECTED_ALL_FEATURES} all-feature predictors, "
        f"found {df_features.shape[1]}"
    )
if len(significant_genes) != EXPECTED_HISTORICAL_FDR_GENES:
    raise ValueError(
        f"Expected {EXPECTED_HISTORICAL_FDR_GENES} historical FDR genes, "
        f"found {len(significant_genes)}"
    )
if int(publication_gene_mask.sum()) != EXPECTED_DUAL_FILTER_GENES:
    raise ValueError(
        f"Expected {EXPECTED_DUAL_FILTER_GENES} dual-filter genes, "
        f"found {int(publication_gene_mask.sum())}"
    )
expected_shap_rows = EXPECTED_HISTORICAL_FDR_GENES * EXPECTED_ALL_FEATURES
if len(shap_significant_genes) != expected_shap_rows:
    raise ValueError(
        f"Expected {expected_shap_rows} SHAP rows, found {len(shap_significant_genes)}"
    )
if set(shap_significant_genes["gene"]) != set(significant_genes):
    raise ValueError("SHAP genes do not match the historical FDR-selected set")

validation_summary = pd.DataFrame([
    {"check": "cells", "observed": len(cell_metadata_out), "expected": EXPECTED_N_CELLS},
    {"check": "genes", "observed": len(rf_gene_metrics), "expected": N_HVGS},
    {"check": "held_out_predictions", "observed": len(rf_test_predictions), "expected": N_HVGS * EXPECTED_N_CELLS},
    {"check": "historical_fdr_genes", "observed": len(significant_genes), "expected": EXPECTED_HISTORICAL_FDR_GENES},
    {"check": "dual_filter_genes", "observed": int(publication_gene_mask.sum()), "expected": EXPECTED_DUAL_FILTER_GENES},
    {"check": "permutation_null_threshold", "observed": float(null_threshold), "expected": np.nan},
])
validation_summary.to_csv(OUTPUT_DIR / "rerun_validation_summary.csv", index=False)
print(validation_summary.to_string(index=False))
print(f"Validated staged rerun in {OUTPUT_DIR}. No publication files were replaced.")
